In [19]:
from rsa_utils import *
import random
from utils import *
%load_ext autoreload
%autoreload 2

✓ Environment ready
The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


# IND-CPA


> A partire dal ciphertext si possono recuperare solo informazioni trascurabili sul plaintext; un messaggio cifrato non deve rilevare alcuna informazione utile sul messaggio originale.


- La nozione di **semantic security** si puo' tradurre in un attack game (modella solo eavesdropping).   

- Invece di chiedere all'attaccante "hai imparato qualcosa sul messaggio?", gli facciamo affrontare una scelta tra due possibili messaggi.  

- L'obiettivo del gioco e' verificare che l'attaccante $\mathcal{A}$ non sia in grado di comprendere quale plaintext corrisponde al ciphertext che ha ricevuto.

<br>

<div style="display:flex; text-align:center; justify-content: center; gap: 30px;">
  <div style="width: 500px; text-align: center;">
    <img src="imgs/SemanticSecurity.svg" style="width: 500px; height: auto;">
  </div>

</div>

In [20]:
"""
Dati i seguenti parametri verificare se RSA è IND-CPA secure
- random.randint(1,n) per pescare un intero a caso 
"""
p = 10037
q = 10007
n = p*q
e = 65537

m0 = 99
m1 = 100



## Come risolviamo il problema?

Il problema di RSA è che ha una cifratura **deterministica**, quindi occorre renderla probabilistica.  

### OAEP

Grazie a questa conversione la cifratura diventa probabilistica e riusciamo a garantire IND-CCA2

<div style="display:flex; text-align:center; justify-content: center; gap: 30px;">
  <div style="width: 500px; text-align: center;">
    <img src="imgs/OAEP.svg" style="width: 500px; height: auto;">
  </div>

</div>

In [23]:
# Parametri
K = 1024
K_BYTES = 1024 // 8
K0_BYTES = 128 // 8  # 16 byte
K1_BYTES = 128 // 8  # 16 byte
N_MAX = K_BYTES - K0_BYTES - K1_BYTES

# Funzioni G e H disponibli
# G(seed)
# H(msg_padded)

def oaep_encode(msg):
    msg_bytes = msg.encode('utf-8')
    if len(msg_bytes) > N_MAX:
        print("Spezzare")
    # MSG TO INT AND PADDING
    msg_padded = (
        msg_bytes
        + b'\x00' * (N_MAX - len(msg_bytes))
        + b'\x00' * K1_BYTES
    )
    print("=" * 70)
    print("Messaggio Paddato")
    print("=" * 70)
    print(msg_padded)
    r = secrets.token_bytes(K0_BYTES)
    # OUTPUT 
    X = xor_bytes(msg_padded, G(r))
    Y = xor_bytes(r, H(X))
    em = X + Y
    return int.from_bytes(em ,byteorder='big')



msg = "ciao mi chiamo davide e ho partecipato ad avanti un altro"

print("=" * 70)
print("Messaggio Originale")
print("=" * 70)
print(msg.encode('utf-8'))
print()

msg_1 = oaep_encode(msg)
print() 

print("=" * 70)
print("Messaggio dopo OAEP")
print("=" * 70)
print(msg_1)

Messaggio Originale
b'ciao mi chiamo davide e ho partecipato ad avanti un altro'

Messaggio Paddato
b'ciao mi chiamo davide e ho partecipato ad avanti un altro\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00'

Messaggio dopo OAEP
63896975946505618945383872648680564539709394350601200088167396868966037070389546489339839360265358796926637863814505708415315148064434619080950960524063507944605683640708033951400392422869213727609708030084949031025143535699256505803481284611015813611741895586732249159855663401605802376841468243801495214916
